In [1]:
import pandas as pd
import numpy as np
import glob, os, shutil
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize as MplNormalize


def getFiles(path, limit=None, shuffle=False):
    target = sorted(glob.glob(os.path.join(path, '*')))
    if shuffle:
        np.random.shuffle(target) 
    return target[:limit]

def formatAxis(img):
    return np.transpose(img, (0, 2, 1))

def setFolder(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)

def showTile(img=None, mask=None, save=None):
    if img is None and mask is None:
        return print("Erro: Forneça pelo menos 'img' ou 'mask'.")

    ref_vol = img if img is not None else mask
    mid_x = ref_vol.shape[0] // 2
    mid_y = ref_vol.shape[1] // 2
    mid_z = ref_vol.shape[2] // 2

    def get_slices(vol):
        if vol is None:
            return None
        
        s_x = np.array(vol[mid_x, :, :]) # Plano YZ
        s_y = np.array(vol[:, mid_y, :]) # Plano XZ
        s_z = np.array(vol[:, :, mid_z]) # Plano XY
        return [s_x, np.rot90(s_z, -1), s_y]

    img_slices  = get_slices(img)
    mask_slices = get_slices(mask)
    cmap_mask_only    = ListedColormap(['black', 'red', 'green', 'blue'])
    cmap_mask_overlay = ListedColormap([(0, 0, 0, 0), 'red', 'green', 'blue'])

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    titles    = [f'Slice X={mid_x}', f'Slice Y={mid_y}', f'Slice Z={mid_z}']

    for i, ax in enumerate(axes):
        if img is not None:
            ax.imshow(img_slices[i], cmap='gray')
            
        if mask is not None:
            if img is not None:
                ax.imshow(mask_slices[i], cmap=cmap_mask_overlay, vmin=0, vmax=3, alpha=0.6)
            else:
                ax.imshow(mask_slices[i], cmap=cmap_mask_only, vmin=0, vmax=3)
        
        ax.set_title(titles[i])

    plt.tight_layout()

    if save:
        plt.savefig(save, bbox_inches='tight', dpi=300)
        return plt.close(fig)

    plt.show()


def show3DCube(ax, volume, label, x_ratio=0.1, y_ratio=0.9, z_ratio=0.9, stride=1):
    nx, ny, nz = volume.shape
    pos_x, pos_y, pos_z = int(nx * x_ratio), int(ny * y_ratio), int(nz * z_ratio)

    cmap = plt.cm.gray
    norm = MplNormalize(vmin=volume.min(), vmax=volume.max())

    def plot_plane(axis_to_fix, fixed_pos):
        if axis_to_fix == 'y':    # Plano XZ
            ranges_dim1 = [(0, pos_x + 1), (pos_x, nx)]
            ranges_dim2 = [(0, pos_z + 1), (pos_z, nz)]
        elif axis_to_fix == 'x':  # Plano YZ
            ranges_dim1 = [(0, pos_y + 1), (pos_y, ny)]
            ranges_dim2 = [(0, pos_z + 1), (pos_z, nz)]
        else:                     # Plano XY (z)
            ranges_dim1 = [(0, pos_x + 1), (pos_x, nx)]
            ranges_dim2 = [(0, pos_y + 1), (pos_y, ny)]

        for start1, end1 in ranges_dim1:
            for start2, end2 in ranges_dim2:
                arr1, arr2 = np.arange(start1, end1), np.arange(start2, end2)

                if axis_to_fix == 'y':
                    X, Z = np.meshgrid(arr1, arr2, indexing='ij')
                    Y = np.full_like(X, fixed_pos)
                    Z_plot = nz - Z
                    data = volume[start1:end1, fixed_pos, start2:end2]
                elif axis_to_fix == 'x':
                    Y, Z = np.meshgrid(arr1, arr2, indexing='ij')
                    X = np.full_like(Y, fixed_pos)
                    Z_plot = nz - Z
                    data = volume[fixed_pos, start1:end1, start2:end2]
                else:
                    X, Y = np.meshgrid(arr1, arr2, indexing='ij')
                    Z_plot = np.full_like(X, nz - fixed_pos)
                    data = volume[start1:end1, start2:end2, fixed_pos]

                ax.plot_surface(X, Y, Z_plot, facecolors=cmap(norm(data)), shade=False, antialiased=False, linewidth=0, rstride=stride, cstride=stride)

    plot_plane('y', pos_y) # Parede XZ
    plot_plane('x', pos_x) # Parede YZ
    plot_plane('z', pos_z) # Chão XY

    ax.set_xlim(0, nx)
    ax.set_ylim(0, ny)
    ax.set_zlim(0, nz)
    ax.set_box_aspect([1, 1, 1])
    ax.set_axis_off()
    ax.set_title(label, fontsize=14, fontweight='bold', loc='left')
    ax.view_init(elev=20, azim=-45)


def showSteps(steps, save=None):
    fig = plt.figure(figsize=(18, 12))
    for i, (volume, label) in enumerate(steps):
        ax = fig.add_subplot(2, 3, i + 1, projection='3d')
        show3DCube(ax, volume, label)
        
    plt.tight_layout()

    if save:
        plt.savefig(save, bbox_inches='tight', dpi=300)

    plt.show()

In [2]:
import numpy as np
from tqdm import tqdm
import scipy.ndimage as ndimage
import os, json


class SyntheticGenerator:
    def __init__(self, shape=(128, 128, 128)):
        # ── Image Format ─────────────────────────────────────────────
        self.margin = 64                  # Buffer para absorver dobras extremas nas bordas com segurança
        self.finalShape = shape           # (nx, ny, nz) final output volume size

        # ── Refletividade (Estratigrafia) ────────────────────────────
        self.layerRange = (100, 230)      # Qtd de camadas. ↑ Imagem cheia de linhas finas. ↓ Blocos grossos e lisos.
        self.layerThickness = (1, 2)      # Espessura. ↑ Camadas mais grossas. ↓ Camadas bem fininhas.

        # ── Dobramentos (Folding) ────────────────────────────────────
        self.foldCount = (15, 30)         # Qtd de dobras. ↑ Imagem muito ondulada. ↓ Terreno plano.
        self.foldSigma = (8, 44)          # Largura da dobra. ↑ Dobras largas e suaves. ↓ Dobras curtas e apertadas.
        self.foldAmplitude = (-17, 17)    # Altura da dobra. ↑ Picos e vales extremos. ↓ Dobras rasas.
        self.foldDamping   = 1.5          # Perda de força. ↑ A dobra some rápido no fundo. ↓ A dobra desce até a base.
        self.foldBaseShift = (-1.6, 1.6)  # Posição Z. ↑/↓ Sobe ou desce o desenho inteiro na imagem.

        # ── Cisalhamento / Inclinação (Shearing) ─────────────────────
        self.shearOffset   = (-2.8, 2.8)  # Deslocamento lateral. ↑/↓ Empurra todo o bloco para o lado.
        self.shearGradient = (-0.1, 0.1)  # Inclinação (Mergulho). ↑ Camadas ficam na diagonal. ↓ Ficam na horizontal.

        # ── Falhas (Faulting) ────────────────────────────────────────
        self.faultCount = (4, 7)          # Qtd de falhas. ↑ Imagem toda fraturada. ↓ Imagem mais inteira.
        self.faultThrow = (0, 22)         # Tamanho do degrau. ↑ Desencontro gigante nas linhas. ↓ Quebra quase invisível.
        self.faultDipAngle = (20, 75)     # Ângulo. ↑ Falha quase em pé (vertical). ↓ Falha deitada.
        
        self.faultRoughness  = 3.3        # Textura do corte. ↑ Corte tremido/áspero. ↓ Corte liso como navalha.
        self.faultRoughSigma = 4.5        # Tamanho da tremedeira. ↑ Ondas grandes na falha. ↓ Ondinhas curtas.
        self.faultDecaySigma = (33, 83)   # Arrasto. ↑ A linha entorta muito antes de quebrar. ↓ Quebra seca.

        self.faultZoneWidth  = 1.2        # Espessura do rótulo. ↑ A máscara da falha fica grossa. ↓ Fica fina.
        self.faultThreshold  = 0.8        # Filtro de rótulo. ↑ Marca só falha grande. ↓ Marca qualquer rachadurazinha.

        self.faultCurveProb  = 0.30       # Chance de curvar. ↑ Falha faz formato de colher (lístrica). ↓ Falha reta.
        self.faultCurveMax   = 6.7        # Força da curva. ↑ Curva muito fechada. ↓ Curva leve.

        # ── Assinatura Sísmica (Wavelet) ─────────────────────────────
        self.waveletFreq = (81, 117)      # Resolução. ↑ Imagem super nítida. ↓ Imagem borrada e grossa.
        self.waveletDuration = 0.08       # "Eco" do sinal. ↑ O traço borra verticalmente. ↓ Sinal limpo e curto.
        self.waveletDt = 0.002            # Amostragem. ↑ Imagem pode ficar pixelada/serrilhada. ↓ Imagem contínua.

        # ── Ruído Final (Noise) ──────────────────────────────────────
        self.noiseLevel = (0.00, 0.10)    # Chuvisco. ↑ Imagem cheia de ruído (ruim). ↓ Imagem limpa (perfeita).

        self.nx = self.finalShape[0] + 2 * self.margin
        self.ny = self.finalShape[1] + 2 * self.margin
        self.nz = self.finalShape[2] + 2 * self.margin
        self.shape = (self.nx, self.ny, self.nz)

    def get(self):
        data = self.genReflectivity()
        data = self.applyFolding(data)
        data = self.applyShearing(data)
        data, mask = self.applyFaulting(data)
        image = self.applyWavelet(data)
        image = self.applyNoise(image)

        image = self.crop(image)
        mask  = self.crop(mask)
        image = (image - np.mean(image)) / (np.std(image) + 1e-8)
        return image.astype(np.float32), mask.astype(np.uint8)

    def set(self, options):
        for k, v in options.items():
            setattr(self, k, v)

    def _generate_single(self, args):
        import numpy as np
        import os
        
        i, imgDir, mskDir, seed = args
        np.random.seed(seed)
        image, mask = self.get()
        image, mask = np.transpose(image, (0, 2, 1)), np.transpose(mask, (0, 2, 1))
        
        np.save(os.path.join(imgDir, f"img_{i:04d}.npy"), image)
        np.save(os.path.join(mskDir, f"img_{i:04d}.npy"), mask)

    def dataset(self, n=200, outputDir="output", n_jobs=None):
        from Utils.index import setFolder
        import concurrent.futures
        import multiprocessing
        import os
        from tqdm import tqdm
        
        imgDir = os.path.join(outputDir, "images")
        mskDir = os.path.join(outputDir, "masks")
        setFolder(imgDir)
        setFolder(mskDir)

        if n_jobs is None:
            n_jobs = multiprocessing.cpu_count()
            
        base_seed = np.random.randint(0, 1000000)
        tasks = [(i, imgDir, mskDir, base_seed + i) for i in range(n)]
        
        with concurrent.futures.ProcessPoolExecutor(max_workers=n_jobs) as executor:
            list(tqdm(executor.map(self._generate_single, tasks), total=n, desc="Generating dataset"))

    def genReflectivity(self):
        """Create 1D layered reflectivity tiled across the volume."""
        r1d = np.zeros(self.nz)
        nLayers = np.random.randint(*self.layerRange)

        for _ in range(nLayers):
            pos = np.random.randint(0, self.nz)
            thickness = np.random.randint(*self.layerThickness)
            r1d[pos : pos + thickness] = np.random.uniform(-1, 1)

        return np.tile(r1d, (self.nx, self.ny, 1))
        
    def applyFolding(self, reflectivity):
        """Deform layers with rotated anisotropic Gaussian folds."""
        x = np.arange(self.nx)
        y = np.arange(self.ny)
        xx, yy = np.meshgrid(x, y, indexing="ij")

        a0 = np.random.uniform(*self.foldBaseShift)
        nGaussians = np.random.randint(*self.foldCount)
        shift2d    = np.zeros((self.nx, self.ny))

        for _ in range(nGaussians):
            x0 = np.random.uniform(-self.nx * 0.3, self.nx * 1.3)
            y0 = np.random.uniform(-self.ny * 0.3, self.ny * 1.3)
            sigmaX = np.random.uniform(*self.foldSigma)
            sigmaY = np.random.uniform(*self.foldSigma)
            theta  = np.random.uniform(0, np.pi)
            amp = np.random.uniform(*self.foldAmplitude)

            dx = xx - x0
            dy = yy - y0
            cosT, sinT = np.cos(theta), np.sin(theta)
            u = cosT * dx + sinT * dy
            v = -sinT * dx + cosT * dy
            shift2d += amp * np.exp(-(u**2 / (2 * sigmaX**2) + v**2 / (2 * sigmaY**2)))

        zGrid = np.arange(self.nz)
        damping = self.foldDamping * zGrid / (self.nz - 1)
        s1 = a0 + shift2d[:, :, np.newaxis] * damping

        ix, iy, iz = np.indices(self.shape)
        return ndimage.map_coordinates(reflectivity, [ix, iy, iz + s1], order=3, mode="nearest")

    def applyShearing(self, reflectivity):
        """Apply linear shear (dip/tilt) along X and Y axes."""
        e0 = np.random.uniform(*self.shearOffset)
        f  = np.random.uniform(*self.shearGradient)
        g  = np.random.uniform(*self.shearGradient)

        ix, iy, iz = np.indices(self.shape)
        s2 = e0 + f * ix + g * iy
        return ndimage.map_coordinates(reflectivity, [ix, iy, iz + s2], order=3, mode="nearest")

    def applyFaulting(self, reflectivity):
        """Inject faults with displacement and produce binary mask."""
        masks = np.zeros(self.shape, dtype=np.uint8)
        model = np.copy(reflectivity)

        numFaults  = np.random.randint(*self.faultCount)
        ix, iy, iz = np.indices(self.shape)

        for i in range(numFaults):
            p0 = np.random.uniform(0.15, 0.85, 3) * np.array(self.shape)

            dip_angle  = np.random.uniform(*self.faultDipAngle)
            dip_rad    = np.deg2rad(dip_angle)
            strike_rad = np.random.uniform(0, 2 * np.pi)
            nx = np.sin(dip_rad) * np.cos(strike_rad)
            ny = np.sin(dip_rad) * np.sin(strike_rad)
            nz = np.cos(dip_rad) * np.random.choice([-1.0, 1.0])
            normal = np.array([nx, ny, nz])

            strike = np.array([-normal[1], normal[0], 0.0])
            strikeNorm = np.linalg.norm(strike)
            strike = np.array([1.0, 0.0, 0.0]) if strikeNorm < 1e-6 else strike / strikeNorm
            dip = np.cross(normal, strike)
            dip /= np.linalg.norm(dip)

            dx = ix - p0[0]
            dy = iy - p0[1]
            dz = iz - p0[2]

            distStrike = strike[0] * dx + strike[1] * dy + strike[2] * dz
            distDip = dip[0] * dx + dip[1] * dy + dip[2] * dz
            bend    = 0.0
            
            if np.random.random() < self.faultCurveProb:
                max_dist = max(self.shape) / 1.5 
                intensidade_base = np.random.uniform(self.faultCurveMax * 0.5, self.faultCurveMax)
                direcao = np.random.choice([-1.0, 1.0])
                curve_intensity = intensidade_base * direcao
                bend = curve_intensity * ((distDip / max_dist) ** 2)

            noisePlane = ndimage.gaussian_filter(np.random.normal(0, 1, self.shape), sigma=self.faultRoughSigma) * self.faultRoughness
            distPlane  = normal[0] * dx + normal[1] * dy + normal[2] * dz + noisePlane - bend
            maxDisp  = np.random.uniform(*self.faultThrow)
            throwMap = self.computeThrowMap(distStrike, distDip, maxDisp)

            hw = distPlane > 0
            throw_hw = throwMap[hw]

            ixShifted = ix.astype(np.float32)
            iyShifted = iy.astype(np.float32)
            izShifted = iz.astype(np.float32)
            
            ixShifted[hw] += throw_hw * dip[0]
            iyShifted[hw] += throw_hw * dip[1]
            izShifted[hw] += throw_hw * dip[2]

            model = ndimage.map_coordinates(model, [ixShifted, iyShifted, izShifted], order=1, mode="nearest")
            masks = ndimage.map_coordinates(masks, [ixShifted, iyShifted, izShifted], order=0, mode="constant", cval=0)
            faultZone = (np.abs(distPlane) <= self.faultZoneWidth) & (np.abs(throwMap) > self.faultThreshold)
            masks[faultZone] = 1

        return model, masks

    def computeThrowMap(self, distStrike, distDip, maxDisp):
        """Compute displacement map for a single fault (gaussian or linear decay)."""
        if np.random.random() < 0.5:
            sigmaPlane = np.random.uniform(*self.faultDecaySigma)
            return maxDisp * np.exp(-(distStrike**2 + distDip**2) / (2 * sigmaPlane**2))

        planeExtent = np.sqrt(self.nx**2 + self.ny**2 + self.nz**2)
        direction   = np.random.choice([-1, 1])
        return maxDisp * np.clip(0.5 + direction * distDip / planeExtent, 0, 1)

    def applyWavelet(self, model):
        """Convolve with a Ricker wavelet along the Z axis."""
        f = np.random.uniform(*self.waveletFreq)
        t = np.arange(-self.waveletDuration, self.waveletDuration, self.waveletDt)
        wavelet = (1 - 2 * (np.pi * f * t) ** 2) * np.exp(-((np.pi * f * t) ** 2))
        return ndimage.convolve1d(model, wavelet, axis=2)

    def applyNoise(self, image):
        """Add band-limited Gaussian noise scaled to signal amplitude."""
        scale = np.random.uniform(*self.noiseLevel) * np.std(image)
        noise = np.random.normal(0.0, 1.0, image.shape)
        noise = ndimage.gaussian_filter(noise, sigma=(1.0, 1.0, 0.5))
        noise *= (scale / (np.std(noise) + 1e-8))
        
        image = (image + noise)
        image = ndimage.gaussian_filter(image, sigma=(0.5, 0.5, 0))
        return image

    def crop(self, volume):
        """Removes the safety margin to extract the final shape volume."""
        x0, x1 = self.margin, self.nx - self.margin
        y0, y1 = self.margin, self.ny - self.margin
        z0, z1 = self.margin, self.nz - self.margin
        return volume[x0:x1, y0:y1, z0:z1]

    def getMetrics(self):
        return {
            "shape": self.shape,
            "margin": self.margin,
            "layerRange": self.layerRange,
            "layerThickness": self.layerThickness,
            "foldCount": self.foldCount,
            "foldSigma": self.foldSigma,
            "foldAmplitude": self.foldAmplitude,
            "foldDamping": self.foldDamping,
            "foldBaseShift": self.foldBaseShift,
            "shearOffset": self.shearOffset,
            "shearGradient": self.shearGradient,
            "faultCount": self.faultCount,
            "faultThrow": self.faultThrow,
            "faultDipAngle": self.faultDipAngle,
            "faultRoughness": self.faultRoughness,
            "faultRoughSigma": self.faultRoughSigma,
            "faultDecaySigma": self.faultDecaySigma,
            "faultZoneWidth": self.faultZoneWidth,
            "faultThreshold": self.faultThreshold,
            "faultCurveProb": self.faultCurveProb,
            "faultCurveMax": self.faultCurveMax,
            "waveletFreq": self.waveletFreq,
            "waveletDuration": self.waveletDuration,
            "waveletDt": self.waveletDt,
            "noiseLevel": self.noiseLevel
        }
    
    def print(self):
        print(json.dumps(self.getMetrics(), indent=4))

In [1]:
import os, cv2, json, glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import sys, gc
sys.path.append("..")
from Network.index import ModelNetwork
from email import generator
import os
import json
import glob
import pickle
import shutil
import argparse
import gc
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import sys
from Network.index import ModelNetwork

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
HAS_OPTUNA = True

In [2]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

gc.collect()
print(torch.__version__)              # versão do PyTorch
print(torch.cuda.is_available())      # True se detectou a GPU
print(torch.cuda.get_device_name(0))  # nome da GPU

2.7.1+cu118
True
Quadro P6000


In [5]:
def getImage(base_path, file_name=None):
    path = os.path.join(base_path, file_name) if file_name else base_path
    return np.load(path)

class SyntheticDataset(Dataset):
    def __init__(self, img_paths, mask_paths, normalizer=None):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        # O argumento 'normalizer' foi mantido apenas para não quebrar 
        # a chamada atual no SyntheticOptimizer.
        assert len(self.img_paths) == len(self.mask_paths)
    
    def __len__(self):
        return len(self.img_paths)
    
    def __getitem__(self, idx):
        img  = getImage(self.img_paths[idx]).astype(np.float32)
        mask = getImage(self.mask_paths[idx]).astype(np.uint8)
        
        img = (img - np.mean(img)) / (np.std(img) + 1e-8)
        
        img_tensor  = torch.tensor(img, dtype=torch.float32).unsqueeze(0)
        mask_tensor = torch.tensor(mask, dtype=torch.long).unsqueeze(0)
        return img_tensor, mask_tensor

class PercentileNormalizer:
    def __init__(self):
        self.p01 = None
        self.p99 = None
        self.is_fitted = False
    
    def fit(self, arrays):
        """Computa percentis a partir de lista de arrays."""
        stacked = np.concatenate([arr.flatten() for arr in arrays])
        self.p01 = np.percentile(stacked, 1)
        self.p99 = np.percentile(stacked, 99)
        self.is_fitted = True
        print(f"[Normalizer] Percentis: p01={self.p01:.4f}, p99={self.p99:.4f}")
    
    def normalize(self, img):
        """Aplica clipping e escala a uma imagem."""
        if not self.is_fitted:
            raise RuntimeError("Normalizer não foi fitted. Chame fit() primeiro.")
        
        clipped = np.clip(img, self.p01, self.p99)
        normalized = (clipped - self.p01) / (self.p99 - self.p01 + 1e-8)
        return normalized.astype(np.float32)
    
    def to_dict(self):
        return {'p01': float(self.p01) if self.p01 is not None else None,
                'p99': float(self.p99) if self.p99 is not None else None}
    
    @classmethod
    def from_dict(cls, d):
        norm = cls()
        norm.p01 = d.get('p01')
        norm.p99 = d.get('p99')
        norm.is_fitted = norm.p01 is not None and norm.p99 is not None
        return norm

In [ ]:
class SyntheticDataset(Dataset):
    """Dataset para imagens sintéticas com normalização opcional."""
    
    def __init__(self, img_paths, mask_paths, normalizer=None):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.normalizer = normalizer
        assert len(self.img_paths) == len(self.mask_paths)
    
    def __len__(self):
        return len(self.img_paths)
    
    def __getitem__(self, idx):
        img  = getImage(self.img_paths[idx]).astype(np.float32)
        mask = getImage(self.mask_paths[idx]).astype(np.uint8)
        
        if self.normalizer is not None:
            img = self.normalizer.normalize(img)
        
        img_tensor  = torch.tensor(img, dtype=torch.float32).unsqueeze(0)
        mask_tensor = torch.tensor(mask, dtype=torch.long).unsqueeze(0)
        return img_tensor, mask_tensor


class SyntheticOptimizer:
    """Orquestra otimização de hiperparâmetros para geração de dados sísmicos sintéticos."""
    
    def __init__(self, model_path, synthetic_generator_class, save_dir="synthetic/optimization",
                 device="cuda" if torch.cuda.is_available() else "cpu", seed=42):
        """
        Inicializa o otimizador.
        
        Args:
            model_path: Caminho do modelo pré-treinado (com model.pth e info.json)
            synthetic_generator_class: Classe SyntheticGenerator
            save_dir: Diretório para salvar resultados
            device: "cuda" ou "cpu"
            seed: Seed para reproducibilidade
        """
        self.model_path = model_path
        self.SyntheticGenerator = synthetic_generator_class
        self.save_dir = save_dir
        self.device = device
        self.seed = seed
        
        self.network = None
        self.best_iou = -1.0
        self.best_trial_id = None
        self.trial_count = 0
        self.study = None
        
        self._setup_directories()
        self._set_seeds()
        print(f"[Optimizer] Inicializado | device={device} | save_dir={save_dir}")
    
    def _setup_directories(self):
        """Cria diretórios necessários."""
        os.makedirs(self.save_dir, exist_ok=True)
    
    def _set_seeds(self):
        """Define seeds para reproducibilidade."""
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(self.seed)
    
    def load_model(self):
        """Carrega modelo 3D U-Net pré-treinado."""
        print(f"[Optimizer] Carregando modelo de {self.model_path}...")
        
        info_path = os.path.join(self.model_path, 'info.json')
        if not os.path.exists(info_path):
            raise FileNotFoundError(f"Model info não encontrado: {info_path}")
        
        with open(info_path, 'r', encoding='utf-8') as f:
            model_info = json.load(f)
        
        model_options = model_info.get('model', {})
        print(f"[Optimizer] Opções do modelo:\n{json.dumps(model_options, indent=2)}")
        
        self.network = ModelNetwork(**model_options)
        
        model_file = os.path.join(self.model_path, 'model.pth')
        if not os.path.exists(model_file):
            raise FileNotFoundError(f"Model file não encontrado: {model_file}")
        
        model_data = torch.load(model_file, map_location=self.device)
        self.network.model.load_state_dict(model_data['model'])
        self.network.model.to(self.device)
        self.network.model.eval()
        
        print(f"[Optimizer] Modelo carregado com sucesso | device={self.device}")
    
    def _compute_iou(self, loader):
        """Computa IoU dado um DataLoader."""
        self.network.model.eval()
        self.network.iou.reset()
        
        with torch.no_grad():
            for imgs, masks in loader:
                imgs = imgs.to(self.device)
                masks = masks.to(self.device)
                
                logits = self.network.model(imgs)
                
                if self.network.multiclass:
                    preds = torch.argmax(logits, dim=1)
                    target = masks.squeeze(1) if masks.dim() == 5 else masks
                    self.network.iou.update(preds, target)
                else:
                    preds = (torch.sigmoid(logits) > 0.5).int()
                    self.network.iou.update(preds, masks.int())
        
        iou_value = self.network.iou.compute().item()
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        return iou_value
    
    def generate_and_evaluate(self, config, batch_size=10, trial_id=None):
        """
        Gera dados sintéticos e avalia IoU.
        
        Workflow:
          1. Gera batch de volumes
          2. Computa normalização percentílica
          3. Cria DataLoader com imagens normalizadas
          4. Avalia IoU
          5. Limpa dados temporários
        
        Args:
            config: Dict com hiperparâmetros
            batch_size: Quantidade de volumes a gerar
            trial_id: ID do trial para seeding
        
        Returns:
            Valor de IoU (float)
        """
        temp_dir = os.path.join(self.save_dir, "_temp_trial")
        
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)
        os.makedirs(temp_dir, exist_ok=True)
        
        try:
            if trial_id is not None:
                base_seed = self.seed + trial_id * 1000
            else:
                base_seed = self.seed + np.random.randint(0, 1000000)
            
            print(f"  [Trial {trial_id}] Gerando {batch_size} volumes sintéticos...")
            gen = self.SyntheticGenerator()
            gen.set(config)
            gen.dataset(n=batch_size, outputDir=temp_dir, n_jobs=None)
            
            img_paths = sorted(glob.glob(os.path.join(temp_dir, 'images', '*.npy')))
            mask_paths = sorted(glob.glob(os.path.join(temp_dir, 'masks', '*.npy')))
            
            if not img_paths or not mask_paths:
                print(f"  [Trial {trial_id}] Nenhum dado gerado. Pulando.")
                return -1.0
            
            print(f"  [Trial {trial_id}] Computando percentis de normalização...")
            arrays = [np.load(path) for path in img_paths]
            normalizer = PercentileNormalizer()
            normalizer.fit(arrays)
            del arrays
            gc.collect()
            
            dataset = SyntheticDataset(img_paths, mask_paths, normalizer)
            loader = DataLoader(dataset, batch_size=1, shuffle=False, 
                              num_workers=0, pin_memory=False)
            
            print(f"  [Trial {trial_id}] Avaliando IoU...")
            iou_value = self._compute_iou(loader)
            print(f"  [Trial {trial_id}] IoU = {iou_value:.6f}")
            
            return iou_value
        
        except Exception as e:
            print(f"  [Trial {trial_id}] Erro: {e}")
            import traceback
            traceback.print_exc()
            return -1.0
        
        finally:
            if os.path.exists(temp_dir):
                shutil.rmtree(temp_dir)
            gc.collect()
    
    def _save_best_result(self, config, iou_value, trial_id, batch_size=10):
        """Salva melhor resultado com config, imagens, máscaras e stats de normalização."""
        best_dir = os.path.join(self.save_dir, f"best_{trial_id}")
        if os.path.exists(best_dir):
            shutil.rmtree(best_dir)
        os.makedirs(best_dir, exist_ok=True)
        
        print(f"  [Optimizer] Salvando melhor resultado em {best_dir}...")
        
        gen = self.SyntheticGenerator()
        gen.set(config)
        gen.dataset(n=batch_size, outputDir=best_dir, n_jobs=None)
        
        img_paths = sorted(glob.glob(os.path.join(best_dir, 'images', '*.npy')))
        arrays = [np.load(path) for path in img_paths]
        normalizer = PercentileNormalizer()
        normalizer.fit(arrays)
        del arrays
        gc.collect()
        
        config_data = gen.getMetrics()
        config_data['iou'] = float(iou_value)

        with open(os.path.join(best_dir, 'config.json'), 'w') as f:
            json.dump(config_data, f, indent=4)
        
        with open(os.path.join(best_dir, 'normalizer.json'), 'w') as f:
            json.dump(normalizer.to_dict(), f, indent=4)
        
        print(f"  [Optimizer] Melhor resultado salvo.")
    
    def _objective(self, trial):
        """Função objetivo para Optuna."""
        trial_id = trial.number
        print(f"\n[Trial {trial_id}] Iniciando otimização...")
        
        try:
            config = self._sample_config(trial)
            iou_value = self.generate_and_evaluate(config, batch_size=10, trial_id=trial_id)
            
            if iou_value < 0:
                return 0.0
            
            if iou_value > self.best_iou:
                self.best_iou = iou_value
                self.best_trial_id = trial_id
                self._save_best_result(config, iou_value, trial_id, batch_size=10)
                print(f"[Optimizer] NOVO MELHOR IoU: {iou_value:.6f} (Trial {trial_id})")
            
            self.trial_count += 1
            return iou_value
        
        except Exception as e:
            print(f"[Trial {trial_id}] Exceção durante trial: {e}")
            import traceback
            traceback.print_exc()
            return 0.0
    
    def _sample_config(self, trial):
        """Amostra configuração do espaço de hiperparâmetros usando Optuna."""
        config = {
            'layerRange': (
                trial.suggest_int('layer_min', 50, 150),
                trial.suggest_int('layer_max', 200, 450)
            ),
            'layerThickness': (
                trial.suggest_int('thick_min', 1, 3),
                trial.suggest_int('thick_max', 3, 10)
            ),
            'foldCount': (
                trial.suggest_int('fold_cnt_min', 4, 20),
                trial.suggest_int('fold_cnt_max', 25, 50)
            ),
            'foldSigma': (
                trial.suggest_int('fold_sig_min', 3, 22),
                trial.suggest_int('fold_sig_max', 23, 80)
            ),
            'foldAmplitude': (
                trial.suggest_int('fold_amp_min', -35, 0),
                trial.suggest_int('fold_amp_max', 0, 45)
            ),
            'foldDamping': trial.suggest_float('fold_damping', 0.0, 10.0),
            'foldBaseShift': (
                -trial.suggest_float('fold_shift_neg', 0.0, 4.0),
                trial.suggest_float('fold_shift_pos', 0.0, 4.0)
            ),
            
            'shearOffset': (
                -trial.suggest_float('shear_offset_neg', 0.0, 8.0),
                trial.suggest_float('shear_offset_pos', 0.0, 8.0)
            ),
            'shearGradient': (
                -trial.suggest_float('shear_grad_neg', 0.0, 0.4),
                trial.suggest_float('shear_grad_pos', 0.0, 0.4)
            ),
            'faultCount': (4, 7),
            'faultThrow': (
                trial.suggest_int('fault_thr_min', 0, 8),
                trial.suggest_int('fault_thr_max', 9, 45)
            ),
            'faultDipAngle': (
                trial.suggest_int('dip_min', 10, 55),
                trial.suggest_int('dip_max', 56, 89)
            ),
            'faultRoughness': trial.suggest_float('fault_rough', 0.0, 10.0),
            'faultRoughSigma': trial.suggest_float('fault_rough_sigma', 0.1, 12.0),
            'faultDecaySigma': (
                trial.suggest_int('fault_decay_min', 1, 60),
                trial.suggest_int('fault_decay_max', 61, 150)
            ),
            'faultZoneWidth': trial.suggest_float('fault_zone_width', 0.5, 3.5),
            'faultThreshold': trial.suggest_float('fault_threshold', 0.01, 2.0),
            'faultCurveProb': trial.suggest_float('fault_curve_prob', 0.0, 0.7),
            'faultCurveMax': trial.suggest_float('fault_curve_max', 0.0, 9.0),
            
            'waveletFreq': (
                trial.suggest_int('wave_freq_min', 15, 100),
                trial.suggest_int('wave_freq_max', 101, 220)
            ),
            'waveletDuration': trial.suggest_float('wavelet_duration', 0.02, 0.20),
            'waveletDt': trial.suggest_float('wavelet_dt', 0.0001, 0.015),
            
            'noiseLevel': (0.0, trial.suggest_float('noise_level_max', 0.0, 0.6)),
        }
        return config
    
    def run(self, n_trials=100, batch_size=10, resume=True):
        """
        Executa loop de otimização.
        
        Args:
            n_trials: Número total de trials
            batch_size: Batch size para cada trial
            resume: Resumir de checkpoint se disponível
        """
        if not HAS_OPTUNA:
            raise RuntimeError("optuna é necessário. Execute: pip install optuna")
        
        print(f"\n{'='*70}")
        print(f"Pipeline de Otimização de Dados Sísmicos Sintéticos")
        print(f"{'='*70}")
        print(f"n_trials={n_trials} | batch_size={batch_size} | device={self.device}")
        
        if self.network is None:
            self.load_model()
        
        study_file = os.path.join(self.save_dir, 'study_state.pkl')
        
        if resume and os.path.exists(study_file):
            print(f"\n[Optimizer] Resumindo de checkpoint: {study_file}")
            with open(study_file, 'rb') as f:
                study_data = pickle.load(f)
                
            self.study = study_data['study']
            self.best_iou = study_data.get('best_iou', -1.0)
            self.best_trial_id = study_data.get('best_trial_id', None)
            self.trial_count = study_data.get('trial_count', 0)
            
            current_trials = len(self.study.trials)
            remaining_trials = max(0, n_trials - current_trials)
            print(f"[Optimizer] Trials atuais: {current_trials} | Rodando {remaining_trials} mais...")
            n_trials = remaining_trials
        else:
            print(f"\n[Optimizer] Iniciando estudo de otimização novo...")
            sampler = TPESampler(seed=self.seed)
            pruner = MedianPruner(n_startup_trials=5)
            self.study = optuna.create_study(
                direction='maximize',
                sampler=sampler,
                pruner=pruner
            )
        
        try:
            self.study.optimize(self._objective, n_trials=n_trials, show_progress_bar=True)
        except KeyboardInterrupt:
            print("\n[Optimizer] Otimização interrompida pelo usuário.")
        
        self._save_checkpoint()
        self._print_summary()
    
    def _save_checkpoint(self):
        """Salva estado do otimizador para resumição."""
        checkpoint = {
            'study': self.study,
            'best_iou': self.best_iou,
            'best_trial_id': self.best_trial_id,
            'trial_count': self.trial_count,
        }
        checkpoint_file = os.path.join(self.save_dir, 'study_state.pkl')
        with open(checkpoint_file, 'wb') as f:
            pickle.dump(checkpoint, f)
        print(f"[Optimizer] Checkpoint salvo: {checkpoint_file}")
        
        df_trials = self.study.trials_dataframe()
        csv_file = os.path.join(self.save_dir, 'search_log.csv')
        df_trials.to_csv(csv_file, index=False)
        print(f"[Optimizer] Histórico salvo: {csv_file}")
    
    def _print_summary(self):
        """Imprime resumo da otimização."""
        print(f"\n{'='*70}")
        print(f"Resumo da Otimização")
        print(f"{'='*70}")
        print(f"Total de trials: {len(self.study.trials)}")
        print(f"Melhor IoU: {self.best_iou:.6f}")
        if self.best_trial_id is not None:
            print(f"Melhor trial: {self.best_trial_id}")
            best_dir = os.path.join(self.save_dir, f"best_{self.best_trial_id}")
            print(f"Melhor resultado salvo em: {best_dir}")
        
        df_trials = self.study.trials_dataframe()
        df_sorted = df_trials.sort_values('value', ascending=False)
        print(f"\nTop 5 trials:")
        print(df_sorted[['number', 'value']].head().to_string(index=False))
        
        print(f"\nSearch log: {os.path.join(self.save_dir, 'search_log.csv')}")
        print(f"{'='*70}\n")


optimizer = SyntheticOptimizer('../Model/Backup/model_1', SyntheticGenerator)
optimizer.run(n_trials=5000, batch_size=10)

[Optimizer] Inicializado | device=cuda | save_dir=synthetic/optimization

Pipeline de Otimização de Dados Sísmicos Sintéticos
n_trials=5000 | batch_size=10 | device=cuda
[Optimizer] Carregando modelo de ../Model/Backup/model_1...
[Optimizer] Opções do modelo:
{
  "network": "standard",
  "img_size": [
    128,
    128,
    128
  ],
  "classes": 1,
  "channels": 1,
  "dropout": 0.1,
  "num_filters": 16,
  "lr": 0.001
}


[I 2026-06-04 18:25:34,811] A new study created in memory with name: no-name-5a09f484-c48b-463f-9b46-f0985e4499a3


[Optimizer] Modelo carregado com sucesso | device=cuda

[Optimizer] Iniciando estudo de otimização novo...


  0%|          | 0/5000 [00:00<?, ?it/s]


[Trial 0] Iniciando otimização...
  [Trial 0] Gerando 10 volumes sintéticos...


...

  [Trial 641] Computando percentis de normalização...
[Normalizer] Percentis: p01=-2.2261, p99=2.2086
  [Trial 641] Avaliando IoU...
  [Trial 641] IoU = 0.329295
[I 2026-06-05 09:55:36,355] Trial 641 finished with value: 0.32929468154907227 and parameters: {'layer_min': 125, 'layer_max': 228, 'thick_min': 2, 'thick_max': 3, 'fold_cnt_min': 17, 'fold_cnt_max': 37, 'fold_sig_min': 13, 'fold_sig_max': 23, 'fold_amp_min': -32, 'fold_amp_max': 0, 'fold_damping': 1.1178594701114741, 'fold_shift_neg': 1.7378506174618051, 'fold_shift_pos': 1.3895170766661744, 'shear_offset_neg': 6.751349097357477, 'shear_offset_pos': 2.0489165822277653, 'shear_grad_neg': 0.3556587293757275, 'shear_grad_pos': 0.2579471885859283, 'fault_thr_min': 1, 'fault_thr_max': 21, 'dip_min': 52, 'dip_max': 81, 'fault_rough': 3.9341403524040177, 'fault_rough_sigma': 5.972821653759486, 'fault_decay_min': 21, 'fault_decay_max': 120, 'fault_zone_width': 2.751864480818159, 'fault_threshold': 0.10467906674485505, 'fault_curve_pr

Generating dataset: 100%|██████████| 10/10 [01:21<00:00,  8.19s/it]


  [Trial 642] Computando percentis de normalização...
[Normalizer] Percentis: p01=-2.2000, p99=2.2398
  [Trial 642] Avaliando IoU...
  [Trial 642] IoU = 0.733300
[I 2026-06-05 09:57:00,642] Trial 642 finished with value: 0.7332999110221863 and parameters: {'layer_min': 124, 'layer_max': 235, 'thick_min': 2, 'thick_max': 3, 'fold_cnt_min': 17, 'fold_cnt_max': 40, 'fold_sig_min': 21, 'fold_sig_max': 23, 'fold_amp_min': -31, 'fold_amp_max': 29, 'fold_damping': 0.9678108862498798, 'fold_shift_neg': 1.4570008818335092, 'fold_shift_pos': 1.3100331999682482, 'shear_offset_neg': 7.029255942778185, 'shear_offset_pos': 2.3921701238748123, 'shear_grad_neg': 0.3478941844655478, 'shear_grad_pos': 0.25004411661390236, 'fault_thr_min': 0, 'fault_thr_max': 22, 'dip_min': 51, 'dip_max': 82, 'fault_rough': 4.042249129859032, 'fault_rough_sigma': 6.144741951181785, 'fault_decay_min': 19, 'fault_decay_max': 123, 'fault_zone_width': 1.0200304348226636, 'fault_threshold': 0.16936996910887134, 'fault_curve_p

Generating dataset: 100%|██████████| 10/10 [01:22<00:00,  8.23s/it]


  [Trial 643] Computando percentis de normalização...
[Normalizer] Percentis: p01=-2.1095, p99=2.1672
  [Trial 643] Avaliando IoU...
  [Trial 643] IoU = 0.748085
[I 2026-06-05 09:58:25,823] Trial 643 finished with value: 0.7480848431587219 and parameters: {'layer_min': 127, 'layer_max': 244, 'thick_min': 2, 'thick_max': 3, 'fold_cnt_min': 17, 'fold_cnt_max': 41, 'fold_sig_min': 21, 'fold_sig_max': 24, 'fold_amp_min': -32, 'fold_amp_max': 28, 'fold_damping': 1.6869655263810521, 'fold_shift_neg': 1.823857901344795, 'fold_shift_pos': 1.379213395473517, 'shear_offset_neg': 7.4882531091696185, 'shear_offset_pos': 2.1258574576884803, 'shear_grad_neg': 0.3474255386687443, 'shear_grad_pos': 0.27023402828865234, 'fault_thr_min': 2, 'fault_thr_max': 31, 'dip_min': 51, 'dip_max': 81, 'fault_rough': 3.8210897855676595, 'fault_rough_sigma': 6.400262114622125, 'fault_decay_min': 20, 'fault_decay_max': 119, 'fault_zone_width': 1.0286164254791634, 'fault_threshold': 0.1951709676060103, 'fault_curve_pr